In [27]:
import os, glob, json
import unicodedata
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

# DATA_PATH = '/kaggle/input/tashkeel-dataset/dataset/'
DATA_PATH = 'dataset/'
TRAIN_FILE = os.path.join(DATA_PATH, 'train.txt')
VAL_FILE = os.path.join(DATA_PATH, 'val.txt')
OUTPUT_MODEL_PATH = '/kaggle/working/bilstm_diac_pytorch_with_der.pt'

# Hyperparameters
MAXLEN = 500
EMBED_DIM = 25
LSTM_UNITS = 256
FF_UNITS = 512
DROPOUT = 0.5
BATCH_SIZE = 256
EPOCHS = 50
PLACEHOLDER = '<NONAR>'
PAD_TOKEN = '<PAD>'
SOS_TOKEN = '<SOS>'
EOS_TOKEN = '<EOS>'
UNK_TOKEN = '<UNK>'
SPACE_TOKEN = '<SPACE>'


Using device: cpu


In [ ]:
# Check if a character is a combining diacritic
def is_combining(ch):
    return unicodedata.category(ch) == 'Mn'

# Split a string into (base_char, diacritics) pairs
def split_char_diacritic_pairs(sentence):
    pairs = []
    base = None
    diacs = ''
    for ch in sentence:
        if is_combining(ch):
            if base is None:
                base = '<UNK_BASE>'
            diacs += ch
        else:
            if base is not None:
                pairs.append((base, diacs))
            base = ch
            diacs = ''
    # Avoid losing the last pair
    if base is not None:
        pairs.append((base, diacs))
    return pairs

# Check if a character is an Arabic letter
def is_arabic_letter(ch):
    # Check if ch is a single character
    if not isinstance(ch, str) or len(ch) != 1:
        return False
    # Converts the character into its Unicode code point
    code = ord(ch)
    return (
        (0x0600 <= code <= 0x06FF) or
        (0x0750 <= code <= 0x077F) or
        (0x08A0 <= code <= 0x08FF) or
        (0xFB50 <= code <= 0xFDFF) or
        (0xFE70 <= code <= 0xFEFF)
    )

# Transform (base_char, diacritics) pairs with placeholders
def placeholder_transform_pairs(pairs, placeholder=PLACEHOLDER):
    tokens = []
    labels = []
    for base, d in pairs:
        # Handle <UNK_BASE>
        if isinstance(base, str) and base.startswith('<') and base.endswith('>'):
            tokens.append(base)
            labels.append('')
        # Handle space
        elif base.isspace():
            tokens.append(SPACE_TOKEN)
            labels.append('')
        # Handle Arabic letters and Tatweel
        elif is_arabic_letter(base) or base == 'ـ':
            tokens.append(base)
            labels.append(d)
        # Handle other non-Arabic characters as Numbers
        else:
            tokens.append(placeholder)
            labels.append('')
    return tokens, labels


In [ ]:
class BiLSTM_Diac(nn.Module):
    def __init__(self, vocab_size, emb_dim, lstm_units, ff_units, num_labels, pad_idx=0, dropout=0.5):
        super().__init__()
        # Map input character indices to dense embeddings, ensuring padding_idx is not updated during training
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)

        # BiDirectional LSTM layers, with dropout for regularization (Avoid overfitting & memorization by randomly dropping units)
        self.bilstm1 = nn.LSTM(emb_dim, lstm_units, num_layers=1, batch_first=True, bidirectional=True)
        self.dropout1 = nn.Dropout(dropout)
        self.bilstm2 = nn.LSTM(2*lstm_units, lstm_units, num_layers=1, batch_first=True, bidirectional=True)
        self.dropout2 = nn.Dropout(dropout)

        # Feedforward layers to mix LSTM outputs to diacritic label logits
        self.ff1 = nn.Linear(2*lstm_units, ff_units)
        self.ff2 = nn.Linear(ff_units, ff_units)
        self.out = nn.Linear(ff_units, num_labels)
        self.relu = nn.ReLU()
    # Forward pass
    def forward(self, x, lengths=None):
        emb = self.embedding(x)
        if lengths is not None:
            # Pack padded sequence for efficient processing by LSTM
            packed = nn.utils.rnn.pack_padded_sequence(emb, lengths.cpu(), batch_first=True, enforce_sorted=False)
            packed_out1, _ = self.bilstm1(packed)
            # Unpack the sequence back to padded form
            out1, _ = nn.utils.rnn.pad_packed_sequence(packed_out1, batch_first=True)
        else:
            # Directly pass embeddings through the first BiLSTM layer, Used in validation when all sequences are of same length
            out1, _ = self.bilstm1(emb)
        out1 = self.dropout1(out1)
        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(out1, lengths.cpu(), batch_first=True, enforce_sorted=False)
            packed_out2, _ = self.bilstm2(packed)
            out2, _ = nn.utils.rnn.pad_packed_sequence(packed_out2, batch_first=True)
        else:
            out2, _ = self.bilstm2(out1)
        out2 = self.dropout2(out2)
        # Feedforward layers with ReLU activations
        ff = self.relu(self.ff1(out2))
        ff = self.relu(self.ff2(ff))
        logits = self.out(ff)
        return logits

In [ ]:
# Inference using trained model

# Load trained model (weights + vocab)
MODEL_PATH = "Output/bilstm_diac_pytorch_with_der.pt"  

ckpt = torch.load(MODEL_PATH, map_location=device)

# checkpoint was saved like {"model_state_dict": ..., "char2idx": ..., "diac2idx": ...}
char2idx = ckpt["char2idx"]
diac2idx = ckpt["diac2idx"]
idx2diac = {v: k for k, v in diac2idx.items()}  


vocab_size = len(char2idx)
num_labels = len(diac2idx)
pad_idx = char2idx[PAD_TOKEN]


model = BiLSTM_Diac(
    vocab_size=vocab_size,
    emb_dim=EMBED_DIM,
    lstm_units=LSTM_UNITS,
    ff_units=FF_UNITS,
    num_labels=num_labels,
    pad_idx=pad_idx,
    dropout=DROPOUT,
).to(device)

# Load model weights
model.load_state_dict(ckpt["model_state_dict"])
# Set model to evaluation mode
model.eval()

# Convert raw line to tokens and track original non-Arabic chars
def prepare_inference_raw_line(raw_line, placeholder=PLACEHOLDER):
    tokens = []
    original_nonar = []
    for ch in raw_line:
        if is_arabic_letter(ch) or ch == "ـ":
            tokens.append(ch)
        elif ch.isspace():
            tokens.append(SPACE_TOKEN)
            original_nonar.append((len(tokens) - 1, ch))
        else:
            tokens.append(placeholder)
            original_nonar.append((len(tokens) - 1, ch))
    return tokens, original_nonar


# Map predicted label to diacritic mark(s)
def label_to_mark(lab):
    # Treat '<NONE>', <OTHER> or empty as no diacritic
    if not lab or lab == "<NONE>" or lab == "<OTHER>":
        return ""
    # If all chars are combining marks, keep them
    if all(unicodedata.category(ch) in ("Mn") for ch in lab):
        return lab
    # Otherwise just return label
    return lab

def infer_and_reconstruct(raw_line):
    # Prepare tokens
    tokens, original_nonar = prepare_inference_raw_line(raw_line)
    seq_mod = [SOS_TOKEN] + tokens + [EOS_TOKEN]

    x_ids = [char2idx.get(t, char2idx[UNK_TOKEN]) for t in seq_mod]
    if len(x_ids) < MAXLEN:
        x_ids = x_ids + [char2idx[PAD_TOKEN]] * (MAXLEN - len(x_ids))
    else:
        x_ids = x_ids[:MAXLEN]

    x_tensor = torch.tensor([x_ids], dtype=torch.long).to(device)
    lengths = torch.tensor([min(len(seq_mod), MAXLEN)], dtype=torch.long).to(device)

    with torch.no_grad():
        logits = model(x_tensor, lengths)
        pred_ids = torch.argmax(logits, dim=-1).cpu().numpy()[0]

    pred_diacs = [idx2diac.get(int(i), "<NONE>") for i in pred_ids]
    # Skip SOS and align with tokens
    pred_trim = pred_diacs[1 : 1 + len(tokens)]

    # build diacritized string token-by-token
    out_tokens = []
    for i, tok in enumerate(tokens):
        if tok == SPACE_TOKEN:
            out_tokens.append(" ")
            continue
        if tok == PLACEHOLDER:
            # Restore original non-Arabic character if available
            repl = next((ch for pos, ch in original_nonar if pos == i), PLACEHOLDER)
            out_tokens.append(repl)
            continue
        # Normal Arabic char
        lab = pred_trim[i] if i < len(pred_trim) else "<NONE>"
        mark = label_to_mark(lab)
        combined = unicodedata.normalize("NFC", tok + mark)
        out_tokens.append(combined)

    return "".join(out_tokens)

# Testing on test set
test_file_path = os.path.join("", 'test_no_diacritics.txt')
raw_text = ""
if os.path.exists(test_file_path):
    with open(test_file_path, 'r', encoding='utf8') as f:
        for _ in range(20):
            line = f.readline()
            if not line:
                break
            raw_text += line

for line in raw_text.splitlines():
    line = line.strip()
    if not line:
        print()
        continue
    print("RAW:  ", line)
    print("DIAC: ", infer_and_reconstruct(line))
    print()


C:\Users\menna\AppData\Local\Temp\ipykernel_89444\3467645936.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(MODEL_PATH, map_location=device)


RAW:   ليس للوكيل بالقبض أن يبرأ المدين أو يهب الدين له أو يأخذ رهنا من المدين في مقابل الدين أو يقبل إحالته على شخص آخر لكن له أن يأخذ كفيلا لكن ليس له أن يأخذ كفيلا بشرط براءة الأصيل انظر المادة ( 648 ) ( الأنقروي ، الطحطاوي وصرة الفتاوى ، البحر ) .
DIAC:  لَيْسَ لِلْوَكِيلِ بِالْقَبْضِ أَنْ يَبْرَأَ الْمَدِينَ أَوْ يَهَبَ الدَّيْنَ لَهُ أَوْ يَأْخُذَ رَهْنًا مِنْ الْمَدِينِ فِي مُقَابِلِ الدَّيْنِ أَوْ يَقْبَلَ إحَالَتُهُ عَلَى شَخْصٍ آخَرَ لَكِنْ لَهُ أَنْ يَأْخُذَ كَفِيلًا لَكِنْ لَيْسَ لَهُ أَنْ يَأْخُذَ كَفِيلًا بِشَرْطِ بَرَاءَةِ الْأَصِيلِ اُنْظُرْ الْمَادَّةَ ( 648 ) ( الْأَنْقِرْوِيُّ ، الطَّحْطَاوِيُّ وَصُرَّةُ الْفَتَاوَى ، الْبَحْرُ ) .

RAW:   ( قوله ويقع في بعض النسخ بمنفعة ومعين ) أي : أوصى بمجموع شيئين بمنفعة شيء وبمعين وقوله وليس ذلك بصحيح كأن عدم الصحة من جهة أن هذه المسألة فيها نص بهذا الحكم الذي أشار إليه المصنف بقوله وإن أوصى بمنفعة معين وبعض شيوخنا علل عدم الصحة بقوله لما علمت من اختلاف الحكم بين الإيصاء بمنفعة المعين ونفس المعين ووقع التنظير وهو أنه هل من منفعة